In [2]:
# to avoid to restart kernel when external modules are modified
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Dataset

Source: [Star dataset to predict star types](https://www.kaggle.com/datasets/deepu1109/star-dataset)

In [3]:
!pip install pandas


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import pandas as pd

df = pd.read_csv("data/stars.csv")
df

,Temperature (K),Luminosity(L/Lo),Radius(R/Ro),Absolute magnitude(Mv),Star type,Star color,Spectral Class
0,3068,0.002400,0.1700,16.12,0,Red,M
1,3042,0.000500,0.1542,16.60,0,Red,M
2,2600,0.000300,0.1020,18.70,0,Red,M
3,2800,0.000200,0.1600,16.65,0,Red,M
4,1939,0.000138,0.1030,20.06,0,Red,M
...,...,...,...,...,...,...,...
235,38940,374830.000000,1356.0000,-9.93,5,Blue,O
236,30839,834042.000000,1194.0000,-10.63,5,Blue,O
237,8829,537493.000000,1423.0000,-10.73,5,White,A
238,9235,404940.000000,1112.0000,-11.23,5,White,A


In [5]:
df[(df['Temperature (K)'].isna() | df['Radius(R/Ro)'].isna() | df["Absolute magnitude(Mv)"].isna())]

,Temperature (K),Luminosity(L/Lo),Radius(R/Ro),Absolute magnitude(Mv),Star type,Star color,Spectral Class


In [6]:
df["Temperature (K)"].describe()

count      240.000000
mean     10497.462500
std       9552.425037
min       1939.000000
25%       3344.250000
50%       5776.000000
75%      15055.500000
max      40000.000000
Name: Temperature (K), dtype: float64

In [13]:
df["Temperature (K)"].nunique()

228

In [7]:
df['Radius(R/Ro)'].describe()

count     240.000000
mean      237.157781
std       517.155763
min         0.008400
25%         0.102750
50%         0.762500
75%        42.750000
max      1948.500000
Name: Radius(R/Ro), dtype: float64

In [14]:
df['Radius(R/Ro)'].nunique()

216

In [8]:
df['Absolute magnitude(Mv)'].describe()

count    240.000000
mean       4.382396
std       10.532512
min      -11.920000
25%       -6.232500
50%        8.313000
75%       13.697500
max       20.060000
Name: Absolute magnitude(Mv), dtype: float64

In [9]:
df["temperature"] = df["Temperature (K)"].copy()
df["radius"] = df["Radius(R/Ro)"].copy()
df["abs_mag"] = df["Absolute magnitude(Mv)"].copy()

features = ["temperature", "radius"]
target_name = "abs_mag"

examples = df[features + [target_name]].to_dict(orient="records")
examples

[{'temperature': 3068, 'radius': 0.17, 'abs_mag': 16.12},
 {'temperature': 3042, 'radius': 0.1542, 'abs_mag': 16.6},
 {'temperature': 2600, 'radius': 0.102, 'abs_mag': 18.7},
 {'temperature': 2800, 'radius': 0.16, 'abs_mag': 16.65},
 {'temperature': 1939, 'radius': 0.103, 'abs_mag': 20.06},
 {'temperature': 2840, 'radius': 0.11, 'abs_mag': 16.98},
 {'temperature': 2637, 'radius': 0.127, 'abs_mag': 17.22},
 {'temperature': 2600, 'radius': 0.096, 'abs_mag': 17.4},
 {'temperature': 2650, 'radius': 0.11, 'abs_mag': 17.45},
 {'temperature': 2700, 'radius': 0.13, 'abs_mag': 16.05},
 {'temperature': 3600, 'radius': 0.51, 'abs_mag': 10.69},
 {'temperature': 3129, 'radius': 0.3761, 'abs_mag': 11.79},
 {'temperature': 3134, 'radius': 0.196, 'abs_mag': 13.21},
 {'temperature': 3628, 'radius': 0.393, 'abs_mag': 10.48},
 {'temperature': 2650, 'radius': 0.14, 'abs_mag': 11.782},
 {'temperature': 3340, 'radius': 0.24, 'abs_mag': 13.07},
 {'temperature': 2799, 'radius': 0.16, 'abs_mag': 14.79},
 {'tem

In [38]:
from regression_tree import RegressionTree


model = RegressionTree(examples, features, target_name)
model.train(depth=5)

best_split_point={'feature': 'radius', 'value': 1.113, 'mse': 12.211847956477868, 'split_index': 128}
best_split_point={'feature': 'radius', 'value': 0.7625, 'mse': 6.844292847591145, 'split_index': 120}
best_split_point={'feature': 'temperature', 'value': 2986.0, 'mse': 4.856156715385612, 'split_index': 28}
best_split_point={'feature': 'radius', 'value': 0.135, 'mse': 2.39777281547619, 'split_index': 24}
best_split_point={'feature': 'radius', 'value': 0.1, 'mse': 1.8743190972222212, 'split_index': 12}
best_split_point={'feature': 'temperature', 'value': 2724.5, 'mse': 1.2837166666666648, 'split_index': 1}
best_split_point={'feature': 'temperature', 'value': 3546.0, 'mse': 4.117561435050232, 'split_index': 44}
best_split_point={'feature': 'radius', 'value': 0.14, 'mse': 2.339819015113919, 'split_index': 13}
best_split_point={'feature': 'temperature', 'value': 18315.0, 'mse': 1.535656380729166, 'split_index': 40}
best_split_point={'feature': 'temperature', 'value': 6158.0, 'mse': 0.4450

In [39]:
def mse_score(y_true: list, y_pred: list) -> float:
    n = len(y_true)
    return sum((true - pred)**2 for true, pred in zip(y_true, y_pred)) / n

In [40]:
import random

y_gt = list()
y_pred = list()


for example in examples:
    y_gt.append(example["abs_mag"])
    y_pred.append(model.predict(example))

    #print(examples[idx], model.predict(examples[idx]))


mse_score(y_gt, y_pred)

1.2797605959821186

In [41]:
model.print()

{'feature': 'radius', 'value': 1.113, 'mse': 12.211847956477868, 'split_index': 128} 
{'feature': 'radius', 'value': 0.7625, 'mse': 6.844292847591145, 'split_index': 120} {'feature': 'radius', 'value': 11.3, 'mse': 4.623430512109375, 'split_index': 32} 
{'feature': 'temperature', 'value': 2986.0, 'mse': 4.856156715385612, 'split_index': 28} {'feature': 'temperature', 'value': 6158.0, 'mse': 0.44502685714285717, 'split_index': 7} {'feature': 'temperature', 'value': 13574.5, 'mse': 1.797154796875, 'split_index': 14} {'feature': 'radius', 'value': 792.45, 'mse': 1.1671364743433397, 'split_index': 41} 
{'feature': 'radius', 'value': 0.135, 'mse': 2.39777281547619, 'split_index': 24} {'feature': 'temperature', 'value': 3546.0, 'mse': 4.117561435050232, 'split_index': 44} {'feature': 'temperature', 'value': 4819.0, 'mse': 0.09418885714285723, 'split_index': 2} None {'feature': 'temperature', 'value': 8151.0, 'mse': 1.1206174214285713, 'split_index': 4} {'feature': 'temperature', 'value': 221

In [ ]:
!pip install matplotlib

In [1]:
import matplotlib.pyplot as plt
import numpy as np

temps = [e["temperature"] for e in examples]
radii = [e["radius"] for e in examples]
actual = [e["abs_mag"] for e in examples]
predicted = [model.predict(e) for e in examples]

fig = plt.figure(figsize=(16, 6))

# --- 3D scatter: temperatura + radio (log) + magnitud real vs predicha ---
ax1 = fig.add_subplot(121, projection="3d")
ax1.scatter(temps, np.log10(radii), actual, c=actual, cmap="plasma", s=20, alpha=0.7, label="Real")
ax1.scatter(temps, np.log10(radii), predicted, c=predicted, cmap="plasma", s=20, alpha=0.4, marker="*", label="Predicción")
ax1.set_xlabel("Temperatura (K)")
ax1.set_ylabel("log₁₀ Radio (R/Ro)")
ax1.set_zlabel("Magnitud absoluta")
ax1.set_title("3D: Temperatura + Radio → Magnitud")
ax1.legend()

# --- Scatter 2D: temperatura vs radio, color = predicción ---
ax2 = fig.add_subplot(122)
sc = ax2.scatter(temps, radii, c=predicted, cmap="plasma", s=30, alpha=0.8)
ax2.set_yscale("log")
ax2.set_xlabel("Temperatura (K)")
ax2.set_ylabel("Radio (R/Ro) — escala log")
ax2.set_title("Temperatura vs Radio\n(color = magnitud predicha)")
plt.colorbar(sc, ax=ax2, label="Magnitud predicha")

plt.tight_layout()
plt.show()


NameError: name 'examples' is not defined